# Internet-speed processing (Ookla)

Processes Ookla internet-speed measurements into national and subnational
download-speed features.

In [ ]:
from datetime import datetime

import geopandas as gp
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from shapely.geometry import Point

In [ ]:
world = gp.read_file('data/ne_10m_admin_0_countries.zip')

In [ ]:
tiles = gp.read_file('data/2024-10-01_performance_fixed_tiles.zip')

In [ ]:
tiles.head()

In [ ]:
len(tiles)

In [ ]:
GDL_shapefiles = gp.read_file('data/GDL Shapefiles V6.3.zip')

In [ ]:
GDL_shapefiles.head()

In [ ]:
!pip install dask-geopandas

In [ ]:
from dask.diagnostics import ProgressBar

In [ ]:
# Convert the GeoDataFrames to Dask GeoDataFrames
tiles_dask = dg.from_geopandas(tiles, npartitions=10)  # Adjust npartitions based on your system
GDL_shapefiles_dask = dg.from_geopandas(GDL_shapefiles, npartitions=10)

# Print an initial message
print("Starting the spatial join operation...")

# Perform the spatial join with 'intersects' as the predicate
global_tiles_dask = tiles_dask.sjoin(GDL_shapefiles_dask, how="inner", predicate='intersects')

# Print a message to indicate the task is moving
print("Spatial join completed. Converting result to Pandas GeoDataFrame...")

with ProgressBar():
    global_tiles = global_tiles_dask.compute()
# Convert the result back to a Pandas GeoDataFrame
global_tiles = global_tiles_dask.compute()

# Print a final message to confirm completion
print("Spatial join operation complete. Data is now available.")

In [ ]:
g = global_tiles[['quadkey',	'avg_d_kbps',	'avg_u_kbps',	'avg_lat_ms',	'tests',	'devices',	'index_right', 'gdlcode', 'continent', 'iso_code']]

In [ ]:
g.to_csv('very_important_subnational_speed.csv')

In [ ]:
GDL_shapefiles.plot()

In [ ]:
global_tiles = gp.sjoin(tiles, GDL_shapefiles, how="inner", op='intersects')